# Model 1: Semantic + Acoustic Features → PHQ-8 / PCL-C Scores

**Goal:** Predict clinical symptom severity from linguistic and acoustic features extracted from DAIC interview transcripts/audio.

**Inputs:**
- 11 semantic/linguistic features (cognitive distortions, affect, social language)
- 94 acoustic features (F0, MFCCs, jitter, shimmer, HNR, pause patterns, etc.)

**Targets:**
- `PHQ8_total` — depression severity score (0–24, regression + binary classification)
- `PCL_total` — PTSD symptom severity score (17–85, regression + binary classification)

**Dataset:** E-DAIC — 155 participants (train/dev split provided)

**Models:** Ridge Regression, Random Forest, Gradient Boosting (regression); Logistic Regression, Random Forest (classification)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict, KFold
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    classification_report, ConfusionMatrixDisplay,
    f1_score, roc_auc_score
)

SEED = 42
np.random.seed(SEED)

## 1. Load and Merge Data

In [ ]:
feats  = pd.read_csv('daic_features_v4.csv')
labels = pd.read_csv('data/edaic/labels/detailed_labels.csv')

# Merge on participant ID
df = feats.merge(labels, left_on='patient_id', right_on='Participant', how='inner')

# Compute total scores from item sums
PHQ_ITEMS = ['PHQ8_1_NoInterest','PHQ8_2_Depressed','PHQ8_3_Sleep','PHQ8_4_Tired',
             'PHQ8_5_Appetite','PHQ8_6_Failure','PHQ8_7_Concentration','PHQ8_8_Psychomotor']
PCL_ITEMS = [c for c in labels.columns if c.startswith('PCL-C_')]

df['PHQ8_total'] = df[PHQ_ITEMS].sum(axis=1)   # 0–24
df['PCL_total']  = df[PCL_ITEMS].sum(axis=1)   # 17–85

# Binary labels (standard clinical cutoffs)
# PHQ-8 >= 10 = moderate/severe depression
# PCL-C >= 44 = probable PTSD (civilian version)
df['Depression_binary'] = (df['PHQ8_total'] >= 10).astype(int)
df['PTSD_binary']       = (df['PCL_total']  >= 44).astype(int)

print(f'Dataset: {len(df)} participants')
print(f'Splits: {df["split"].value_counts().to_dict()}')
print()
print('PHQ8_total:', df['PHQ8_total'].describe().round(2).to_dict())
print('PCL_total: ', df['PCL_total'].describe().round(2).to_dict())
print()
print(f'Depression (PHQ>=10): {df["Depression_binary"].sum()} / {len(df)}')
print(f'PTSD (PCL>=44):       {df["PTSD_binary"].sum()} / {len(df)}')

In [ ]:
# Define feature groups
SEMANTIC_COLS = [c for c in feats.columns if 'patient_mean' in c]
ACOUSTIC_COLS = [c for c in feats.columns if c not in SEMANTIC_COLS and c != 'patient_id']
ALL_COLS      = SEMANTIC_COLS + ACOUSTIC_COLS

print(f'Semantic features:  {len(SEMANTIC_COLS)}')
print(f'Acoustic features:  {len(ACOUSTIC_COLS)}')
print(f'Total features:     {len(ALL_COLS)}')
print(f'NaN in features:    {df[ALL_COLS].isnull().sum().sum()}')

## 2. Exploratory Analysis

## 3. Cross-Validation Setup

We use the official **train/dev split** from E-DAIC for a single train-test evaluation,  
plus 5-fold cross-validation within the training set for hyperparameter selection.

In [ ]:
train_df = df[df['split'] == 'train'].copy()
dev_df   = df[df['split'] == 'dev'].copy()

X_train = train_df[ALL_COLS].values
X_dev   = dev_df[ALL_COLS].values

# Regression targets
y_phq_train = train_df['PHQ8_total'].values
y_pcl_train = train_df['PCL_total'].values
y_phq_dev   = dev_df['PHQ8_total'].values
y_pcl_dev   = dev_df['PCL_total'].values

# Binary classification targets
y_dep_train = train_df['Depression_binary'].values
y_ptsd_train = train_df['PTSD_binary'].values
y_dep_dev   = dev_df['Depression_binary'].values
y_ptsd_dev  = dev_df['PTSD_binary'].values

print(f'Train: {len(train_df)} | Dev: {len(dev_df)}')
print(f'Train depressed: {y_dep_train.sum()} / {len(y_dep_train)}')
print(f'Train PTSD:      {y_ptsd_train.sum()} / {len(y_ptsd_train)}')
print(f'Dev depressed:   {y_dep_dev.sum()} / {len(y_dep_dev)}')
print(f'Dev PTSD:        {y_ptsd_dev.sum()} / {len(y_ptsd_dev)}')

cv5 = KFold(n_splits=5, shuffle=True, random_state=SEED)

## 4. Regression: Predict PHQ-8 and PCL-C Scores

Three models compared:
- **Ridge** — linear, regularised; interpretable coefficients
- **Gradient Boosting** — non-linear, strong baseline
- **Random Forest** — non-linear, feature importances

In [ ]:
def eval_regression(model, X_train, y_train, X_dev, y_dev, label, feature_names):
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_dev)

    r2   = r2_score(y_dev, y_pred)
    mae  = mean_absolute_error(y_dev, y_pred)
    rmse = np.sqrt(mean_squared_error(y_dev, y_pred))

    print(f'  R²={r2:.3f}  MAE={mae:.2f}  RMSE={rmse:.2f}')

    # Feature importance
    m = pipe.named_steps['model']
    if hasattr(m, 'feature_importances_'):
        imp = m.feature_importances_
    elif hasattr(m, 'coef_'):
        imp = np.abs(m.coef_)
    else:
        imp = None

    return {'target': label, 'model': type(model).__name__,
            'R2': r2, 'MAE': mae, 'RMSE': rmse,
            'y_pred': y_pred, 'importance': imp}


reg_models = [
    ('Ridge',             Ridge(alpha=1.0)),
    ('GradientBoosting',  GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED)),
    ('RandomForest',      RandomForestRegressor(n_estimators=200, random_state=SEED)),
]

reg_results = []
for target_name, y_tr, y_dv in [('PHQ-8', y_phq_train, y_phq_dev),
                                  ('PCL-C', y_pcl_train, y_pcl_dev)]:
    print(f'\n── {target_name} ──')
    for mname, model in reg_models:
        print(f'  {mname}:', end=' ')
        r = eval_regression(model, X_train, y_tr, X_dev, y_dv, target_name, ALL_COLS)
        r['model'] = mname
        reg_results.append(r)

In [ ]:
# Scatter plots: predicted vs actual for best model per target
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for row, (target_name, y_tr, y_dv) in enumerate([('PHQ-8', y_phq_train, y_phq_dev),
                                                    ('PCL-C', y_pcl_train, y_pcl_dev)]):
    for col, (mname, model) in enumerate(reg_models):
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler()),
            ('model',   model)
        ])
        pipe.fit(X_train, y_tr)
        y_pred = pipe.predict(X_dev)
        r2 = r2_score(y_dv, y_pred)
        mae = mean_absolute_error(y_dv, y_pred)

        ax = axes[row, col]
        ax.scatter(y_dv, y_pred, alpha=0.7, color='steelblue' if row == 0 else 'darkorange')
        mn, mx = min(y_dv.min(), y_pred.min()), max(y_dv.max(), y_pred.max())
        ax.plot([mn, mx], [mn, mx], 'k--', linewidth=0.8)
        ax.set_xlabel(f'Actual {target_name}')
        ax.set_ylabel(f'Predicted {target_name}')
        ax.set_title(f'{mname}\nR²={r2:.3f}, MAE={mae:.2f}')

plt.suptitle('Regression: Actual vs Predicted (Dev Set)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('regression_scatter.png', dpi=150)
plt.show()

In [ ]:
# Feature importance: top 20 for GradientBoosting (PHQ-8)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (target_name, y_tr) in zip(axes, [('PHQ-8', y_phq_train), ('PCL-C', y_pcl_train)]):
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED))
    ])
    pipe.fit(X_train, y_tr)
    imp = pipe.named_steps['model'].feature_importances_
    top20 = np.argsort(imp)[-20:]

    colors = ['steelblue' if ALL_COLS[i] in ACOUSTIC_COLS else 'tomato' for i in top20]
    ax.barh(range(20), imp[top20], color=colors)
    ax.set_yticks(range(20))
    ax.set_yticklabels(
        [ALL_COLS[i].replace('patient_mean_avg_','sem:').replace('patient_mean_','sem:') for i in top20],
        fontsize=8
    )
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Top 20 Features — {target_name} (GBM)\n(red=semantic, blue=acoustic)')

plt.tight_layout()
plt.savefig('feature_importance_regression.png', dpi=150)
plt.show()

## 5. Classification: Depression and PTSD Binary Labels

PHQ-8 ≥ 10 → depressed; PCL-C ≥ 44 → probable PTSD.

In [ ]:
def eval_classifier(model, X_train, y_train, X_dev, y_dev, label, feature_names):
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('clf',     model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_dev)
    y_prob = pipe.predict_proba(X_dev)[:, 1] if hasattr(pipe, 'predict_proba') else None

    f1  = f1_score(y_dev, y_pred, average='weighted')
    acc = np.mean(y_dev == y_pred)
    auc = roc_auc_score(y_dev, pipe.predict_proba(X_dev)[:, 1]) if hasattr(pipe.named_steps['clf'], 'predict_proba') else np.nan

    print(f'  Acc={acc:.3f}  F1={f1:.3f}  AUC={auc:.3f}')
    print(classification_report(y_dev, y_pred, target_names=['Negative', 'Positive']))

    return {'target': label, 'model': type(model).__name__,
            'accuracy': acc, 'f1_weighted': f1, 'auc': auc, 'y_pred': y_pred}


clf_models = [
    ('LogisticRegression', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)),
    ('RandomForest',       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED)),
    ('DecisionTree',       DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=SEED)),
]

clf_results = []
for target_name, y_tr, y_dv in [('Depression (PHQ>=10)', y_dep_train, y_dep_dev),
                                   ('PTSD (PCL>=44)',       y_ptsd_train, y_ptsd_dev)]:
    print(f'\n── {target_name} ──')
    for mname, model in clf_models:
        print(f'  {mname}:')
        r = eval_classifier(model, X_train, y_tr, X_dev, y_dv, target_name, ALL_COLS)
        r['model'] = mname
        clf_results.append(r)

In [ ]:
# Confusion matrices for best classifier
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, (target_name, y_tr, y_dv, class_names) in enumerate([
    ('Depression (PHQ>=10)', y_dep_train, y_dep_dev, ['Not Depressed', 'Depressed']),
    ('PTSD (PCL>=44)', y_ptsd_train, y_ptsd_dev, ['No PTSD', 'PTSD']),
]):
    for col, (mname, model) in enumerate([
        ('LogisticRegression', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)),
        ('RandomForest',       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED)),
    ]):
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler()),
            ('clf',     model)
        ])
        pipe.fit(X_train, y_tr)
        y_pred = pipe.predict(X_dev)
        ConfusionMatrixDisplay.from_predictions(
            y_dv, y_pred, display_labels=class_names, ax=axes[row, col]
        )
        axes[row, col].set_title(f'{target_name}\n{mname}')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

## 6. Feature Group Ablation

Compare using semantic features only vs. acoustic features only vs. both combined.

In [ ]:
ablation_results = []

model_factory = lambda: GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED)

for feat_name, feat_cols in [
    ('Semantic only', SEMANTIC_COLS),
    ('Acoustic only', ACOUSTIC_COLS),
    ('Semantic + Acoustic', ALL_COLS),
]:
    X_tr = train_df[feat_cols].values
    X_dv = dev_df[feat_cols].values
    for target_name, y_tr, y_dv in [('PHQ-8', y_phq_train, y_phq_dev),
                                      ('PCL-C', y_pcl_train, y_pcl_dev)]:
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler()),
            ('model',   model_factory())
        ])
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_dv)
        ablation_results.append({
            'features': feat_name,
            'target':   target_name,
            'R2':  round(r2_score(y_dv, y_pred), 3),
            'MAE': round(mean_absolute_error(y_dv, y_pred), 2),
        })

ablation_df = pd.DataFrame(ablation_results)
pivot = ablation_df.pivot_table(index='features', columns='target', values=['R2', 'MAE'])
print('Ablation Study — GradientBoosting on Dev Set:')
print(pivot.to_string())

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, target in zip(axes, ['PHQ-8', 'PCL-C']):
    sub = ablation_df[ablation_df['target'] == target]
    ax.bar(sub['features'], sub['R2'], color=['#4C72B0', '#DD8452', '#55A868'])
    ax.set_ylabel('R²')
    ax.set_title(f'Feature Ablation — {target} (R²)')
    ax.set_ylim(min(sub['R2'].min() - 0.1, -0.1), sub['R2'].max() + 0.1)
    ax.axhline(0, color='black', linewidth=0.8)
    for i, (_, row) in enumerate(sub.iterrows()):
        ax.text(i, row['R2'] + 0.01, f"{row['R2']:.3f}", ha='center', fontsize=10)
    ax.set_xticklabels(sub['features'], rotation=15, ha='right')

plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150)
plt.show()

## 7. Export Model 1 Outputs for Model 3

Export predicted PHQ-8 and PCL-C scores (regression) and depression/PTSD probabilities (classification) for all participants, for use in the final ensemble aggregator.

In [ ]:
X_all = df[ALL_COLS].values
y_phq_all  = df['PHQ8_total'].values
y_pcl_all  = df['PCL_total'].values
y_dep_all  = df['Depression_binary'].values
y_ptsd_all = df['PTSD_binary'].values

# Train final models on full training set, predict on all participants
best_reg = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED))
])

best_clf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED))
])

# PHQ-8 predictions
best_reg.fit(X_train, y_phq_train)
phq_pred_all = best_reg.predict(X_all)

# PCL-C predictions
best_reg.fit(X_train, y_pcl_train)
pcl_pred_all = best_reg.predict(X_all)

# Depression probability
best_clf.fit(X_train, y_dep_train)
dep_prob_all = best_clf.predict_proba(X_all)[:, 1]

# PTSD probability
best_clf.fit(X_train, y_ptsd_train)
ptsd_prob_all = best_clf.predict_proba(X_all)[:, 1]

# Compose output
model1_output = pd.DataFrame({
    'patient_id':         df['patient_id'].values,
    'split':              df['split'].values,
    'PHQ8_true':          y_phq_all,
    'PCL_true':           y_pcl_all,
    'PHQ8_pred':          phq_pred_all.round(2),
    'PCL_pred':           pcl_pred_all.round(2),
    'depression_prob':    dep_prob_all.round(4),
    'ptsd_prob':          ptsd_prob_all.round(4),
    # Normalised 0–1 scores for Model 3 aggregation
    'depression_score':   (phq_pred_all.clip(0, 24) / 24).round(4),
    'ptsd_score':         ((pcl_pred_all.clip(17, 85) - 17) / (85 - 17)).round(4),
})

out_path = 'data/edaic/labels/model1_outputs.csv'
model1_output.to_csv(out_path, index=False)
print(f'Model 1 outputs saved to {out_path}')
print(model1_output.head())

## 8. Results Summary

In [ ]:
print('=== REGRESSION (Dev Set) ===')
reg_df = pd.DataFrame([
    {'target': r['target'], 'model': r['model'], 'R2': r['R2'], 'MAE': r['MAE'], 'RMSE': r['RMSE']}
    for r in reg_results
])
print(reg_df.to_string(index=False))

print()
print('=== CLASSIFICATION (Dev Set) ===')
clf_df = pd.DataFrame([
    {'target': r['target'], 'model': r['model'],
     'accuracy': round(r['accuracy'], 3), 'f1': round(r['f1_weighted'], 3), 'auc': round(r['auc'], 3)}
    for r in clf_results
])
print(clf_df.to_string(index=False))